# Clean Treasury Monitor Verification

这个 notebook 用来逐项核验 `target_treasury_monitor_clean` 的三个核心功能：

1. 账户 dashboard：连接 IB，读取目标账户持仓、账户资金、行情、PnL、Greeks，并生成账户风险表。
2. 期权链批量更新：按指定 ZF 底层期货月份，一次性刷新并保存静态合约、期权链快照、monitor frame。
3. 期权链实时监控：发现关注范围内的近月 / 近 DTE / 近价格合约，保持实时订阅，后续刷新只读 ticker 对象。

运行前请确认 TWS 或 IB Gateway 已启动，并打开 API。建议先用 delayed 数据核验链获取，再切到 live。

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import time

import pandas as pd
from IPython.display import display
from ib_async.ib import StartupFetch

from target_treasury_monitor_clean.account_dashboard import fetch_account_dashboard
from target_treasury_monitor_clean.carry_dashboard_sync import validate_carry_dashboard_files, write_carry_dashboard_files
from target_treasury_monitor_clean.chain_batch import refresh_static_chain
from target_treasury_monitor_clean.chain_realtime import LiveChainMonitor
from target_treasury_monitor_clean.future_bars import fetch_future_bars
from target_treasury_monitor_clean.ib_session import ib_connection
from target_treasury_monitor_clean.quality import evaluate_option_chain_data, print_option_chain_quality_report
from target_treasury_monitor_clean.settings import (
    AccountDashboardSettings,
    IBSettings,
    LiveChainSettings,
    StaticChainSettings,
    market_data_type_from_label,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

## 0. 参数区

只需要先改这里。`MARKET_DATA_TYPE` 可用 `live`、`frozen`、`delayed`、`delayed_frozen`，也可以直接填 `1/2/3/4`。

In [ ]:
HOST = "127.0.0.1"
PORT = 4001
CLIENT_ID = 7351
TARGET_ACCOUNT = "U16251798"  # TODO: 填你的 IB 账户，例如 U1234567
MARKET_DATA_TYPE = "delayed"  # 核验链条时建议先 delayed；账户实盘监控再改 live

OUTPUT_DIR = Path("data/clean_verify")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SHARED_TREASURY_CHAIN_MONTHS = {
    "ZF": "202609,202612",
    "ZN": "202609,202612",
}
ZC_CHAIN_MONTHS = {
    # "ZC": "202609,202612",  # ZC 参数后续单独确认后填这里
}
TREASURY_CHAIN_MONTHS = {
    **SHARED_TREASURY_CHAIN_MONTHS,
    **{root: months for root, months in ZC_CHAIN_MONTHS.items() if str(months).strip()},
}
DASHBOARD_PRODUCTS = ["ZF", "ZN", "ZC"]
FUTURE_BAR_CONTRACTS = [(root, months.split(",")[0]) for root, months in TREASURY_CHAIN_MONTHS.items()]
FUTURE_BAR_SIZE = "30 mins"
FUTURE_BAR_DURATION = "1 M"
FUTURE_BAR_TIMEOUT = 45.0
FUTURE_BAR_CACHE_DIR = OUTPUT_DIR
FUTURE_BAR_PREFER_LOCAL_SYMBOL = False
MIN_EXPIRATION = ""  # 空字符串表示从今天开始
MAX_EXPIRATION = ""  # 空字符串表示不限制

ib_settings = IBSettings(
    host=HOST,
    port=PORT,
    client_id=CLIENT_ID,
    account=TARGET_ACCOUNT.strip(),
    market_data_type=market_data_type_from_label(MARKET_DATA_TYPE),
)

ib_settings

## 1. 连接 IB 并查看可见账户

如果这里连不上，后面的单元格都不用继续。常见原因是 TWS/Gateway 没开 API、端口不对、client id 冲突。

In [ ]:
fetch_fields = StartupFetch.POSITIONS | StartupFetch.ACCOUNT_UPDATES | StartupFetch.SUB_ACCOUNT_UPDATES

with ib_connection(ib_settings, fetch_fields=fetch_fields) as ib:
    print("connected:", ib.isConnected())
    print("managed accounts:", ib.managedAccounts())

## 2. 核验账户 dashboard

目标：读取账户持仓、账户资金、持仓行情、PnL、Greeks，并形成可检查的风险表。

检查点：

- `visible_accounts` 包含你的目标账户。
- `account_summary` 里有 `NetLiquidation`、`ExcessLiquidity`、`AvailableFunds` 等字段。
- `position_frame` 有目标美债期货/期权持仓，且 `price`、`marketValue`、`unrealizedPnL`、`delta/gamma/theta/vega` 尽量不是空。
- `missingData` 用来定位行情、Greeks 或 PnL 缺失原因。

In [ ]:
if not ib_settings.account:
    raise ValueError("请先在参数区填写 TARGET_ACCOUNT")

dashboard_settings = AccountDashboardSettings(
    quote_wait_seconds=6.0,
    infer_spreads=False,
    reference_root="ZF",
)

with ib_connection(ib_settings, fetch_fields=fetch_fields) as ib:
    account_snapshot = fetch_account_dashboard(ib, ib_settings, dashboard_settings)
    print("visible accounts:", account_snapshot.visible_accounts)
    print("treasury position rows:", len(account_snapshot.position_frame))
    print("all position rows:", len(account_snapshot.account_positions))
    print("future reference:", account_snapshot.future_reference)

    account_snapshot.position_frame.to_csv(OUTPUT_DIR / "dashboard_treasury_positions.csv", index=False, encoding="utf-8-sig")
    account_snapshot.account_summary.to_csv(OUTPUT_DIR / "dashboard_account_summary.csv", index=False, encoding="utf-8-sig")
    account_snapshot.greek_summary.to_csv(OUTPUT_DIR / "dashboard_greek_summary.csv", index=False, encoding="utf-8-sig")

In [ ]:
display(account_snapshot.account_summary.sort_values("tag"))
display(account_snapshot.greek_summary)

risk_cols = [
    "optionName", "localSymbol", "secType", "position", "expiry", "strike", "right",
    "bid", "ask", "mid", "last", "price", "priceSource",
    "marketValue", "unrealizedPnL", "iv", "delta", "gamma", "theta", "vega",
    "otmTicks", "moneyness", "missingData", "conId",
]
display(account_snapshot.position_frame[[c for c in risk_cols if c in account_snapshot.position_frame.columns]])

## 3. 核验期权链批量更新

目标：按指定 ZF 底层期货月份，一次性刷新静态合约、期权链快照、monitor frame，并保存到本地。

检查点：

- `contract_count` 大于 0。
- 如果 `request_market_data=True`，`snapshot_count` 和 `monitor_frame` 应有数据。
- 默认会在订阅前过滤：一周以内只保留距离期货价 1 点以内，一周以上保留 3 点以内；全量有效合约仍保存在 contracts CSV。
- 输出路径里应生成 contracts / chain_summary / future_prices / snapshot / monitor_frame CSV。
- 第二次运行相同 `root/months/min_expiration/max_expiration/output_dir` 时，`universe source` 应显示 `contract_cache`，表示已跳过重复合约发现和 qualify。
- 如果需要重新确认有效合约，把 `force_rebuild_contract_cache=True`。

In [ ]:
static_results = {}

with ib_connection(ib_settings) as ib:
    ib.reqMarketDataType(ib_settings.market_data_type)
    for root, months in TREASURY_CHAIN_MONTHS.items():
        static_settings = StaticChainSettings(
            root=root,
            future_months=months,
            min_expiration=MIN_EXPIRATION.strip() or None,
            max_expiration=MAX_EXPIRATION.strip() or None,
            qualify_batch_size=300,
            batch_size=150,
            wait_max_seconds=8,
            wait_stable_seconds=1.5,
            request_interval=0.025,
            inter_batch_pause_seconds=1.0,
            empty_batch_retries=1,
            empty_batch_retry_pause_seconds=5.0,
            request_market_data=True,
            output_dir=OUTPUT_DIR,
            filter_market_data_by_moneyness=True,
            near_dte_days=7,          # DTE <= 7: 只订阅期货价格 1 点以内
            near_strike_width=1.0,
            far_strike_width=3.0,     # DTE > 7: 只订阅期货价格 3 点以内
            use_contract_cache=True,              # 已有 *_contracts.csv 时跳过合约发现/qualify
            force_rebuild_contract_cache=False,   # 改 True 才会强制重新发现/qualify 合约
        )
        print(f"\n[{root}] refreshing static chain: {months}")
        static_results[root] = refresh_static_chain(ib, static_settings)

for root, result in static_results.items():
    print(f"\n[{root}]")
    print("contracts:", result.raw.get("contract_count"))
    print("snapshot rows:", result.raw.get("snapshot_count"))
    print("selected contracts:", result.raw.get("selected_contract_count"))
    print("snapshot scope:", result.raw.get("snapshot_scope"))
    print("universe source:", result.raw.get("universe_source"))
    print("contract cache:", result.raw.get("contract_cache_path"))
    print("saved paths:")
    for name, path in result.saved_paths.items():
        print(f"  {name}: {path}")

combined_static_monitor_frame = pd.concat(
    [result.monitor_frame for result in static_results.values() if not result.monitor_frame.empty],
    ignore_index=True,
) if static_results else pd.DataFrame()
combined_static_monitor_frame

In [ ]:
for root, result in static_results.items():
    print(f"\n[{root}] future prices")
    display(result.raw.get("future_prices"))
    print(f"[{root}] chain summary")
    display(result.raw.get("chain_summary"))
    display(result.metadata.head(20))
    display(result.raw.get("selected_metadata", pd.DataFrame()).head(20))

display(combined_static_monitor_frame.head(30))

carry_dashboard_paths = write_carry_dashboard_files(
    account_snapshot.position_frame,
    combined_static_monitor_frame,
    output_dir=Path("data"),
)
print("carry dashboard sync paths:")
for name, path in carry_dashboard_paths.items():
    print(f"  {name}: {path}")

display(validate_carry_dashboard_files("data"))

In [ ]:
quality_report = evaluate_option_chain_data(combined_static_monitor_frame)
print_option_chain_quality_report(quality_report)

display(quality_report["by_dte"])
display(quality_report["by_distance"])
display(quality_report["samples"]["tiny_ask"])
display(quality_report["samples"]["both_bid_ask_missing"])
display(quality_report["samples"]["near_missing_greeks"])

for root, result in static_results.items():
    if not result.monitor_frame.empty:
        print(f"\n[{root}] quality")
        print_option_chain_quality_report(evaluate_option_chain_data(result.monitor_frame))


## 4. 核验期货 K 线

目标：为 HTML 顶部大图生成 ZF/ZN/ZC 近一个月 30 分钟 OHLCV；ZC 会在 `ZC_CHAIN_MONTHS` 填好后自动纳入。

In [ ]:
with ib_connection(ib_settings) as ib:
    ib.reqMarketDataType(ib_settings.market_data_type)
    futures_bars = fetch_future_bars(
        ib,
        FUTURE_BAR_CONTRACTS,
        bar_size=FUTURE_BAR_SIZE,
        duration=FUTURE_BAR_DURATION,
        timeout=FUTURE_BAR_TIMEOUT,
        cache_dir=FUTURE_BAR_CACHE_DIR,
        strict=False,
        prefer_local_symbol=FUTURE_BAR_PREFER_LOCAL_SYMBOL,
    )

print("future bar rows:", len(futures_bars))
display(futures_bars.groupby("symbol").tail(5) if not futures_bars.empty else futures_bars)

carry_dashboard_paths = write_carry_dashboard_files(
    account_snapshot.position_frame,
    combined_static_monitor_frame,
    bars=futures_bars,
    output_dir=Path("data"),
)
print("carry dashboard sync paths:")
for name, path in carry_dashboard_paths.items():
    print(f"  {name}: {path}")

display(validate_carry_dashboard_files("data"))

## 5. 核验期权链实时监控

目标：先发现关注范围内的合约并订阅一次，然后多次读取同一批 ticker，验证后续刷新没有重复做全量订阅。

检查点：

- `discovery['qualified_count']` 大于 0。
- 第一次 `start()` 会慢一些，因为要发现合约并发起订阅。
- 后续 `snapshot()` 应明显更快，只是读取已订阅 ticker。
- `readiness.quote_ready / greek_ready` 用于观察行情和 Greeks 到达情况。
- `flow_events` 是基于 volume 差分估算的新增成交事件。

In [ ]:
live_settings = LiveChainSettings(
    root="ZF",
    future_months=TREASURY_CHAIN_MONTHS["ZF"],
    max_dte=14,
    max_expirations=8,
    strikes_each_side=12,
    strike_width=5.0,
    qualify_batch_size=250,
    request_interval=0.025,
    warmup_seconds=4.0,
    stable_seconds=1.0,
    poll_seconds=1.0,
    output_path=OUTPUT_DIR / "live_zf_chain_latest.csv",
    flow_db_path=OUTPUT_DIR / "zf_option_flow.sqlite",
    min_volume_delta=1.0,
)

live_rows = []

with ib_connection(ib_settings) as ib:
    ib.reqMarketDataType(ib_settings.market_data_type)
    with LiveChainMonitor(ib, live_settings) as monitor:
        t0 = time.perf_counter()
        discovery = monitor.start()
        start_seconds = time.perf_counter() - t0
        print("qualified contracts:", discovery.get("qualified_count"))
        print("subscription startup seconds:", round(start_seconds, 2))

        for i in range(5):
            t1 = time.perf_counter()
            snap = monitor.snapshot()
            elapsed = time.perf_counter() - t1
            live_rows.append({
                "iteration": i + 1,
                "snapshot_seconds": elapsed,
                "rows": len(snap.raw_snapshot),
                "quote_ready": snap.readiness.quote_ready,
                "greek_ready": snap.readiness.greek_ready,
                "oi_ready": snap.readiness.oi_ready,
                "volume_ready": snap.readiness.volume_ready,
                "flow_events": len(snap.flow_events),
                "saved": str(snap.output_path or ""),
            })
            time.sleep(live_settings.poll_seconds)

live_check = pd.DataFrame(live_rows)
display(live_check)

In [ ]:
latest_live = pd.read_csv(live_settings.output_path) if live_settings.output_path and live_settings.output_path.exists() else pd.DataFrame()
print("latest live csv:", live_settings.output_path)
display(latest_live.head(30))

if not latest_live.empty:
    other_static = combined_static_monitor_frame[
        combined_static_monitor_frame["symbol"].astype(str).str.upper() != live_settings.root.upper()
    ] if "symbol" in combined_static_monitor_frame.columns else pd.DataFrame()
    live_dashboard_chain = pd.concat([other_static, latest_live], ignore_index=True)
    carry_dashboard_paths = write_carry_dashboard_files(
        account_snapshot.position_frame,
        live_dashboard_chain,
        bars=futures_bars if "futures_bars" in globals() else None,
        output_dir=Path("data"),
    )
    print("carry dashboard sync paths:")
    for name, path in carry_dashboard_paths.items():
        print(f"  {name}: {path}")

html_readiness = validate_carry_dashboard_files(
    "data",
    expected_products=",".join(DASHBOARD_PRODUCTS),
    min_chain_rows=50,
    min_bars_rows=100,
    max_chain_age_hours=24,
    max_bars_age_hours=72,
)
display(html_readiness)

## 5. 结论记录

你可以在这里记录核验结果：

- 账户 dashboard 是否符合预期：
- 批量链刷新是否符合预期：
- 实时链刷新速度是否符合预期：
- 发现的问题或下一步要改的地方：